# 01 · Data overview

**Question:** Which orders have complete delivery data, and what does an order's
lifecycle look like from purchase to delivery?

Why it matters: lateness can only be measured on orders that were delivered.
This notebook decides which orders the rest of the project analyses.

Source: the `order_delivery` view (`sql/01_order_delivery.sql`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

# Notebooks run from notebooks/, so add src/ to the import path.
sys.path.insert(0, str(Path.cwd().parent / "src"))
from db import query

sns.set_theme(style="whitegrid")

## 1. Order status

**Question:** How many orders are in each status, and how many count as delivered?

In [ ]:
status = query("""
    SELECT order_status,
           COUNT(*)                              AS orders,
           COUNT(*) FILTER (WHERE is_delivered)  AS delivered_with_date
    FROM order_delivery
    GROUP BY order_status
    ORDER BY orders DESC
""")
status["share_of_orders"] = (status["orders"] / status["orders"].sum()).round(4)
status

**Your notes:**

- 

## 2. Coverage over time

**Question:** Is the number of orders per month stable, or are some months
too sparse to trust (especially at the start and end of the dataset)?

In [ ]:
monthly = query("""
    SELECT date_trunc('month', order_purchase_timestamp)::date AS month,
           COUNT(*)                                            AS orders,
           COUNT(*) FILTER (WHERE is_delivered)                AS delivered
    FROM order_delivery
    GROUP BY 1
    ORDER BY 1
""")

ax = monthly.plot(x="month", y=["orders", "delivered"], marker="o", figsize=(10, 4))
ax.set(title="Orders per purchase month", xlabel="", ylabel="Orders")
plt.show()
monthly

**Your notes:**

- 

## 3. Lifecycle durations

**Question:** For delivered orders, how many days does each stage take:
approval, handing to the carrier, and carrier to customer?

In [ ]:
stages = query("""
    SELECT EXTRACT(EPOCH FROM order_approved_at             - order_purchase_timestamp)     / 86400 AS purchase_to_approval,
           EXTRACT(EPOCH FROM order_delivered_carrier_date  - order_approved_at)            / 86400 AS approval_to_carrier,
           EXTRACT(EPOCH FROM order_delivered_customer_date - order_delivered_carrier_date) / 86400 AS carrier_to_customer,
           EXTRACT(EPOCH FROM order_delivered_customer_date - order_purchase_timestamp)     / 86400 AS purchase_to_customer
    FROM order_delivery
    WHERE is_delivered
""")
stages.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=stages, orient="h", showfliers=False, ax=ax)
ax.set(title="Days per stage, delivered orders (outliers hidden)", xlabel="Days")
plt.show()

**Your notes:**

- 

## 4. Timestamps out of order

**Question:** Are there delivered orders whose dates happen in an impossible
order? Each row is one rule; `orders` is how many orders break it.

In [ ]:
query("""
    SELECT 'approved before purchase'          AS rule, COUNT(*) FILTER (WHERE order_approved_at             < order_purchase_timestamp)     AS orders FROM order_delivery WHERE is_delivered
    UNION ALL
    SELECT 'carrier handoff before approval',            COUNT(*) FILTER (WHERE order_delivered_carrier_date  < order_approved_at)            FROM order_delivery WHERE is_delivered
    UNION ALL
    SELECT 'customer delivery before carrier handoff',   COUNT(*) FILTER (WHERE order_delivered_customer_date < order_delivered_carrier_date) FROM order_delivery WHERE is_delivered
    UNION ALL
    SELECT 'customer delivery before purchase',          COUNT(*) FILTER (WHERE order_delivered_customer_date < order_purchase_timestamp)     FROM order_delivery WHERE is_delivered
    UNION ALL
    SELECT 'delivered but missing approval or carrier date', COUNT(*) FILTER (WHERE order_approved_at IS NULL OR order_delivered_carrier_date IS NULL) FROM order_delivery WHERE is_delivered
""")

**Your notes:**

- 

## Decisions from this notebook

Write down what you decided and why, e.g. which orders are in scope and how
to treat out-of-order timestamps.

-